# 🎓 Professor's AI Assistant — Google ADK Multi-Agent System

**A hands-on multi-agent demo built with Google ADK**

---

## What this notebook does

This is a **multi-agent system** with a root orchestrator that delegates tasks to four specialised sub-agents:

| Agent | Job |
|---|---|
| 🗂️ **Syllabus Agent** | Reads your uploaded syllabus, answers questions about it |
| 📅 **Timetable Agent** | Reads your uploaded timetable, finds free slots, checks schedules |
| 📝 **Quiz Agent** | Generates quiz questions and MCQs from any topic or your syllabus |
| 📧 **Email Agent** | Drafts professional emails to students or colleagues |
| 🎓 **Orchestrator** | Reads your question and routes it to the right agent(s) |

---

## Before you start

1. Get a free Gemini API key → [aistudio.google.com](https://aistudio.google.com)
2. Add it to **Colab Secrets**: click the 🔑 icon in the left sidebar → name it `GOOGLE_API_KEY` → toggle Notebook access ON
3. Run cells **top to bottom** in order
4. After **Cell 1**, go to **Runtime → Restart session**, then continue from Cell 2

---

## Multi-Agent Architecture

```
                    ┌─────────────────────┐
   Your question ──▶│   Orchestrator      │
                    │   (root_agent)      │
                    └──────┬──────────────┘
                           │ routes to
           ┌───────────────┼───────────────┬───────────────┐
           ▼               ▼               ▼               ▼
    ┌─────────────┐ ┌─────────────┐ ┌─────────────┐ ┌─────────────┐
    │  Syllabus   │ │  Timetable  │ │    Quiz     │ │   Email     │
    │   Agent     │ │   Agent     │ │   Agent     │ │   Agent     │
    └─────────────┘ └─────────────┘ └─────────────┘ └─────────────┘
```

Each sub-agent is wrapped as an `AgentTool` — the orchestrator calls them like tools based on your question.

---
## Cell 1 — Install dependencies

> ⚠️ **After running this cell, go to Runtime → Restart session, then continue from Cell 2.**

In [ ]:
!pip install -q google-adk PyPDF2 python-docx
print("✅ Done. Now go to Runtime → Restart session, then run from Cell 2.")

---
## Cell 2 — API key + imports

In [ ]:
import os
from google.colab import userdata
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

from google.adk.agents import LlmAgent
from google.adk.tools.agent_tool import AgentTool
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner
from google.adk.tools import ToolContext
from google.genai.types import Content, Part

print("✅ Imports ready.")

---
## Cell 3 — Upload your files (optional)

Upload a **syllabus** (PDF, DOCX, or TXT) and/or a **timetable** (PDF, DOCX, or TXT).
If you skip this, the agents will still work — they'll just say no file was uploaded.

In [ ]:
from google.colab import files
import PyPDF2
import docx
import io

# ── Stores for extracted text ───────────────────────────────────────────
SYLLABUS_TEXT   = ""
TIMETABLE_TEXT  = ""

def extract_text_from_file(filename: str, content: bytes) -> str:
    """Extract plain text from PDF, DOCX, or TXT file bytes."""
    ext = filename.lower().split(".")[-1]
    if ext == "pdf":
        reader = PyPDF2.PdfReader(io.BytesIO(content))
        return "\n".join(page.extract_text() or "" for page in reader.pages)
    elif ext == "docx":
        doc = docx.Document(io.BytesIO(content))
        return "\n".join(p.text for p in doc.paragraphs)
    else:  # plain text
        return content.decode("utf-8", errors="ignore")

# ── Upload syllabus ─────────────────────────────────────────────────────
print("📄 Upload your SYLLABUS file (PDF / DOCX / TXT) — or press Cancel to skip")
try:
    uploaded = files.upload()
    if uploaded:
        fname = list(uploaded.keys())[0]
        SYLLABUS_TEXT = extract_text_from_file(fname, uploaded[fname])
        print(f"✅ Syllabus loaded: {fname} ({len(SYLLABUS_TEXT)} chars)")
    else:
        print("⚠️  No syllabus uploaded — agent will work without it.")
except Exception:
    print("⚠️  Syllabus upload skipped.")

print()

# ── Upload timetable ────────────────────────────────────────────────────
print("🗓️  Upload your TIMETABLE file (PDF / DOCX / TXT) — or press Cancel to skip")
try:
    uploaded = files.upload()
    if uploaded:
        fname = list(uploaded.keys())[0]
        TIMETABLE_TEXT = extract_text_from_file(fname, uploaded[fname])
        print(f"✅ Timetable loaded: {fname} ({len(TIMETABLE_TEXT)} chars)")
    else:
        print("⚠️  No timetable uploaded — agent will work without it.")
except Exception:
    print("⚠️  Timetable upload skipped.")

print()
print("Ready — continue to Cell 4.")

---
## Cell 4 — Define tools

Each tool is a plain Python function. The `ToolContext` parameter lets tools read/write session state.

> 📌 **ADK key point:** The docstring is what the LLM reads to decide when to call the tool. Write it clearly.

In [ ]:
# ════════════════════════════════════════════════════════════════
# SYLLABUS TOOLS
# ════════════════════════════════════════════════════════════════

def get_syllabus(tool_context: ToolContext) -> str:
    """Retrieve the full syllabus text that was uploaded by the professor.

    Returns:
        The syllabus content as plain text, or a message if none was uploaded.
    """
    text = tool_context.state.get("syllabus", SYLLABUS_TEXT)
    if not text.strip():
        return (
            "No syllabus has been uploaded yet. "
            "The professor can upload a PDF, DOCX, or TXT file in Cell 3."
        )
    return text


def search_syllabus(topic: str, tool_context: ToolContext) -> str:
    """Search the uploaded syllabus for a specific topic or keyword.

    Args:
        topic: The topic or keyword to search for in the syllabus.

    Returns:
        Relevant lines from the syllabus that mention the topic.
    """
    text = tool_context.state.get("syllabus", SYLLABUS_TEXT)
    if not text.strip():
        return "No syllabus uploaded. Please upload one in Cell 3."
    lines = text.splitlines()
    matches = [l for l in lines if topic.lower() in l.lower()]
    if not matches:
        return f"The topic '{topic}' was not found in the syllabus."
    return f"Found {len(matches)} relevant line(s) about '{topic}':\n" + "\n".join(matches[:20])


# ════════════════════════════════════════════════════════════════
# TIMETABLE TOOLS
# ════════════════════════════════════════════════════════════════

def get_timetable(tool_context: ToolContext) -> str:
    """Retrieve the full timetable that was uploaded by the professor.

    Returns:
        The timetable content as plain text, or a message if none was uploaded.
    """
    text = tool_context.state.get("timetable", TIMETABLE_TEXT)
    if not text.strip():
        return (
            "No timetable has been uploaded. "
            "The professor can upload a PDF, DOCX, or TXT file in Cell 3."
        )
    return text


def find_free_slots(day: str, tool_context: ToolContext) -> str:
    """Find free time slots on a given day from the uploaded timetable.

    Args:
        day: The day to check, e.g. 'Monday', 'Tuesday', 'Wednesday'.

    Returns:
        Lines from the timetable for that day, or all content if day not found.
    """
    text = tool_context.state.get("timetable", TIMETABLE_TEXT)
    if not text.strip():
        return "No timetable uploaded. Please upload one in Cell 3."
    lines = text.splitlines()
    matches = [l for l in lines if day.lower() in l.lower()]
    if not matches:
        return f"No entries for '{day}' found in the timetable. Full timetable:\n" + text
    return f"Timetable entries for {day}:\n" + "\n".join(matches)


# ════════════════════════════════════════════════════════════════
# QUIZ TOOLS
# ════════════════════════════════════════════════════════════════

def save_quiz(title: str, content: str, tool_context: ToolContext) -> str:
    """Save a generated quiz to session state so it can be retrieved later.

    Args:
        title: A short title for the quiz, e.g. 'Python Basics Quiz'.
        content: The full quiz text with questions and answers.

    Returns:
        Confirmation message.
    """
    quizzes = tool_context.state.get("saved_quizzes", [])
    quizzes.append({"title": title, "content": content})
    tool_context.state["saved_quizzes"] = quizzes
    return f"Quiz '{title}' saved. You now have {len(quizzes)} quiz(zes) saved."


def list_saved_quizzes(tool_context: ToolContext) -> str:
    """List all quizzes that have been saved in this session.

    Returns:
        A numbered list of saved quiz titles.
    """
    quizzes = tool_context.state.get("saved_quizzes", [])
    if not quizzes:
        return "No quizzes saved yet."
    return "Saved quizzes:\n" + "\n".join(
        f"{i+1}. {q['title']}" for i, q in enumerate(quizzes)
    )


# ════════════════════════════════════════════════════════════════
# EMAIL TOOLS
# ════════════════════════════════════════════════════════════════

def save_draft(subject: str, body: str, tool_context: ToolContext) -> str:
    """Save a drafted email to session state so the professor can copy it later.

    Args:
        subject: The email subject line.
        body: The full email body text.

    Returns:
        Confirmation that the draft was saved.
    """
    drafts = tool_context.state.get("email_drafts", [])
    drafts.append({"subject": subject, "body": body})
    tool_context.state["email_drafts"] = drafts
    return f"Email draft saved with subject: '{subject}'"


def list_drafts(tool_context: ToolContext) -> str:
    """List all saved email drafts from this session.

    Returns:
        Subject lines of all saved drafts.
    """
    drafts = tool_context.state.get("email_drafts", [])
    if not drafts:
        return "No email drafts saved yet."
    return "Saved drafts:\n" + "\n".join(
        f"{i+1}. {d['subject']}" for i, d in enumerate(drafts)
    )


print("✅ All tools defined:")
tools_list = [
    get_syllabus, search_syllabus,
    get_timetable, find_free_slots,
    save_quiz, list_saved_quizzes,
    save_draft, list_drafts
]
for t in tools_list:
    print(f"   • {t.__name__}()")

---
## Cell 5 — Build the four sub-agents

Each sub-agent is an `LlmAgent` with its own instruction and a focused set of tools.

> 📌 **ADK key point:** Each agent's `description` is what the orchestrator reads to decide which agent to delegate to.

In [ ]:
MODEL = "gemini-2.0-flash"

# ── Sub-Agent 1: Syllabus Agent ─────────────────────────────────────────
syllabus_agent = LlmAgent(
    name="syllabus_agent",
    model=MODEL,
    description=(
        "Handles all questions about the course syllabus. "
        "Use this agent to look up topics, check what is covered, "
        "or summarise the syllabus."
    ),
    instruction=(
        "You are a syllabus assistant for a professor. "
        "Use get_syllabus() to read the full syllabus, or "
        "search_syllabus(topic) to find specific topics. "
        "Always cite the relevant section when answering. "
        "If no syllabus was uploaded, say so clearly and suggest uploading one."
    ),
    tools=[get_syllabus, search_syllabus],
)

# ── Sub-Agent 2: Timetable Agent ────────────────────────────────────────
timetable_agent = LlmAgent(
    name="timetable_agent",
    model=MODEL,
    description=(
        "Handles all questions about schedules, class times, and free slots. "
        "Use this agent to find free time, check when a class is scheduled, "
        "or see the weekly timetable."
    ),
    instruction=(
        "You are a timetable assistant for a professor. "
        "Use get_timetable() to read the full schedule, or "
        "find_free_slots(day) to check a specific day. "
        "Be concise and clear. Suggest good times for office hours or meetings. "
        "If no timetable was uploaded, say so and suggest uploading one."
    ),
    tools=[get_timetable, find_free_slots],
)

# ── Sub-Agent 3: Quiz Agent ─────────────────────────────────────────────
quiz_agent = LlmAgent(
    name="quiz_agent",
    model=MODEL,
    description=(
        "Generates quiz questions, MCQs, and assessments on any topic. "
        "Use this agent when the professor wants to create questions, "
        "a test, or a quiz — either from a topic or from the syllabus."
    ),
    instruction=(
        "You are a quiz and assessment generator for a professor. "
        "When asked to create a quiz, generate clear questions with "
        "answer keys. For MCQs, provide 4 options and mark the correct answer. "
        "Vary difficulty: include easy, medium, and hard questions. "
        "After generating, use save_quiz(title, content) to save it. "
        "Use list_saved_quizzes() to show what has been saved."
    ),
    tools=[save_quiz, list_saved_quizzes, get_syllabus],
)

# ── Sub-Agent 4: Email Agent ────────────────────────────────────────────
email_agent = LlmAgent(
    name="email_agent",
    model=MODEL,
    description=(
        "Drafts professional emails for the professor — to students, "
        "colleagues, or administration. Use this agent for announcements, "
        "reminders, feedback emails, or any written communication."
    ),
    instruction=(
        "You are an email writing assistant for a professor. "
        "Write professional, clear, and appropriately formal emails. "
        "Always include a subject line. Use a warm but professional tone. "
        "After drafting, use save_draft(subject, body) to save it. "
        "Use list_drafts() to show saved drafts."
    ),
    tools=[save_draft, list_drafts],
)

print("✅ Sub-agents created:")
for a in [syllabus_agent, timetable_agent, quiz_agent, email_agent]:
    print(f"   • {a.name}")

---
## Cell 6 — Build the Orchestrator

The orchestrator wraps each sub-agent as an `AgentTool`. It reads the professor's question and decides which specialist to call.

> 📌 **ADK key point:** `AgentTool(agent=...)` turns any agent into a callable tool. The orchestrator uses the sub-agent's `description` field to know when to call it.

In [ ]:
root_agent = LlmAgent(
    name="professor_assistant",
    model=MODEL,
    description="An AI assistant for professors that handles syllabus, timetable, quiz, and email tasks.",
    instruction=(
        "You are a helpful AI assistant for a university professor. "
        "You have four specialised sub-agents available as tools:\n"
        "- syllabus_agent: for syllabus questions, topic lookup, course content\n"
        "- timetable_agent: for schedule, free slots, class timings\n"
        "- quiz_agent: for generating quiz questions, MCQs, assessments\n"
        "- email_agent: for drafting emails to students or colleagues\n\n"
        "Route each question to the most appropriate agent. "
        "For complex questions that need multiple agents (e.g. 'create a quiz based on "
        "my syllabus and email it to students'), call multiple agents in sequence. "
        "Always give a clear, helpful final answer."
    ),
    tools=[
        AgentTool(agent=syllabus_agent),
        AgentTool(agent=timetable_agent),
        AgentTool(agent=quiz_agent),
        AgentTool(agent=email_agent),
    ],
)

print("✅ Orchestrator created:", root_agent.name)
print("   Sub-agents registered as AgentTools:")
print("   • syllabus_agent")
print("   • timetable_agent")
print("   • quiz_agent")
print("   • email_agent")

---
## Cell 7 — Set up Runner & Session

In [ ]:
APP_NAME = "professor_assistant"
USER_ID  = "professor_01"

session_service = InMemorySessionService()
runner = Runner(
    agent=root_agent,
    app_name=APP_NAME,
    session_service=session_service,
)
session = session_service.create_session(
    app_name=APP_NAME,
    user_id=USER_ID,
)

# Pre-load uploaded file content into session state
# so all agents can access it via ToolContext
if SYLLABUS_TEXT:
    session.state["syllabus"] = SYLLABUS_TEXT
    print(f"📄 Syllabus loaded into session state ({len(SYLLABUS_TEXT)} chars)")
if TIMETABLE_TEXT:
    session.state["timetable"] = TIMETABLE_TEXT
    print(f"🗓️  Timetable loaded into session state ({len(TIMETABLE_TEXT)} chars)")

print(f"\n✅ Runner ready. Session ID: {session.id}")

---
## Cell 8 — Helper functions

- `ask(question)` — clean output, just the final answer
- `ask_verbose(question)` — shows which sub-agent was called at each step (great for teaching)

In [ ]:
def ask(question: str) -> None:
    """Ask the professor assistant and print the final response."""
    print(f"\n{'─'*60}")
    print(f"🧑‍🏫 Professor: {question}")
    print(f"{'─'*60}")
    message = Content(role="user", parts=[Part(text=question)])
    for event in runner.run(
        user_id=USER_ID,
        session_id=session.id,
        new_message=message,
    ):
        if event.is_final_response():
            print(f"🤖 Assistant:\n{event.content.parts[0].text.strip()}")


def ask_verbose(question: str) -> None:
    """Ask and show every delegation step — for teaching the multi-agent pattern."""
    print(f"\n{'─'*60}")
    print(f"🧑‍🏫 Professor: {question}")
    print(f"{'─'*60}")
    message = Content(role="user", parts=[Part(text=question)])
    for event in runner.run(
        user_id=USER_ID,
        session_id=session.id,
        new_message=message,
    ):
        if not event.content:
            continue
        for part in event.content.parts:
            if hasattr(part, "function_call") and part.function_call:
                fc = part.function_call
                # Show which sub-agent (or tool) was called
                agent_names = ["syllabus_agent", "timetable_agent", "quiz_agent", "email_agent"]
                if fc.name in agent_names:
                    print(f"  🔀 [delegate] orchestrator → {fc.name}")
                else:
                    print(f"  🔧 [tool call] {fc.name}({dict(fc.args) if fc.args else ''})")
            elif hasattr(part, "function_response") and part.function_response:
                fr = part.function_response
                resp_text = str(fr.response)
                preview = resp_text[:80] + "..." if len(resp_text) > 80 else resp_text
                print(f"  📦 [result    ] {fr.name} → {preview}")
            elif hasattr(part, "text") and part.text and event.is_final_response():
                print(f"🤖 Assistant:\n{part.text.strip()}")


print("✅ ask() and ask_verbose() ready.")

---
## Cell 9 — Demo: Syllabus questions

The orchestrator routes these to `syllabus_agent`.

In [ ]:
# If you uploaded a syllabus, replace the topic with something from your file
ask("Give me a brief summary of the course syllabus.")

In [ ]:
# Try with a topic from your own syllabus, e.g. "machine learning" or "data structures"
ask("Is recursion covered in the syllabus? Which week?")

---
## Cell 10 — Demo: Timetable questions

The orchestrator routes these to `timetable_agent`.

In [ ]:
ask("What does my schedule look like on Monday?")

In [ ]:
ask("When do I have free time this week for student office hours?")

---
## Cell 11 — Demo: Generate a quiz

The orchestrator routes these to `quiz_agent`. Watch it save the quiz automatically.

In [ ]:
ask("Create a 5-question MCQ quiz on Python functions for beginner students.")

In [ ]:
ask("Generate 3 short-answer questions on object-oriented programming.")

In [ ]:
ask("Show me all the quizzes I've saved so far.")

---
## Cell 12 — Demo: Draft an email

The orchestrator routes these to `email_agent`.

In [ ]:
ask(
    "Draft an email to students reminding them that the assignment "
    "deadline is this Friday at 11:59pm and late submissions won't be accepted."
)

In [ ]:
ask(
    "Write an email to my colleague Dr. Sharma asking if she is available "
    "for a 30-minute meeting next week to discuss the curriculum update."
)

---
## Cell 13 — Demo: Multi-agent chaining

These questions require the orchestrator to call **two agents in sequence**.
Use `ask_verbose()` to watch the delegation happen.

In [ ]:
# Orchestrator will call quiz_agent (to generate) then email_agent (to draft)
ask_verbose(
    "Create a short quiz on loops in Python and then draft an email "
    "to students telling them about this quiz."
)

In [ ]:
# Orchestrator will call syllabus_agent then quiz_agent
ask_verbose(
    "Look at my syllabus and generate a quiz on the first topic covered."
)

---
## Cell 14 — Inspect session state

See all the data stored across this session — quizzes, email drafts, and file content.

In [ ]:
current = session_service.get_session(
    app_name=APP_NAME,
    user_id=USER_ID,
    session_id=session.id,
)

print("📦 Session state summary:")
print(f"   Syllabus loaded   : {'Yes' if current.state.get('syllabus') else 'No'}")
print(f"   Timetable loaded  : {'Yes' if current.state.get('timetable') else 'No'}")
print(f"   Quizzes saved     : {len(current.state.get('saved_quizzes', []))}")
print(f"   Email drafts saved: {len(current.state.get('email_drafts', []))}")
print(f"   Total events      : {len(current.events)}")

# Print saved quiz titles
quizzes = current.state.get("saved_quizzes", [])
if quizzes:
    print("\n📝 Saved quiz titles:")
    for i, q in enumerate(quizzes, 1):
        print(f"   {i}. {q['title']}")

# Print saved email subjects
drafts = current.state.get("email_drafts", [])
if drafts:
    print("\n📧 Saved email subjects:")
    for i, d in enumerate(drafts, 1):
        print(f"   {i}. {d['subject']}")

---
## Cell 15 — Retrieve a saved quiz or email draft

Print the full content of any saved quiz or email.

In [ ]:
# Change index to retrieve a different quiz (0 = first, 1 = second, ...)
QUIZ_INDEX = 0

quizzes = session.state.get("saved_quizzes", [])
if quizzes and QUIZ_INDEX < len(quizzes):
    q = quizzes[QUIZ_INDEX]
    print(f"📝 Quiz: {q['title']}")
    print("─" * 50)
    print(q["content"])
else:
    print("No quiz at that index. Run Cell 11 first to generate a quiz.")

In [ ]:
# Change index to retrieve a different email draft (0 = first, 1 = second, ...)
DRAFT_INDEX = 0

drafts = session.state.get("email_drafts", [])
if drafts and DRAFT_INDEX < len(drafts):
    d = drafts[DRAFT_INDEX]
    print(f"📧 Subject: {d['subject']}")
    print("─" * 50)
    print(d["body"])
else:
    print("No draft at that index. Run Cell 12 first to draft an email.")

---
## Cell 16 — Try your own questions

Type anything — the orchestrator will route to the right agent automatically.

In [ ]:
# ✏️ Change this to anything you want to ask
MY_QUESTION = "Create a 3-question quiz on recursion and email it to students."

ask_verbose(MY_QUESTION)

---
## Summary — What this notebook taught

| Concept | Where you saw it |
|---|---|
| `LlmAgent` with tools | Cells 4–5 — each sub-agent has focused tools |
| `AgentTool` wrapping | Cell 6 — sub-agents become tools for the orchestrator |
| Orchestrator routing | Cell 8 — orchestrator reads `description` to decide who to call |
| `ToolContext` + state | Cell 4 — tools save quizzes and emails across turns |
| File upload + extraction | Cell 3 — PDF/DOCX/TXT parsed into session state |
| Multi-agent chaining | Cell 13 — one question, two agents called in sequence |
| Event stream inspection | `ask_verbose()` — every delegation step is visible |

---

## Ideas to extend this

- Add a **Grading Agent** that evaluates student answers against a rubric
- Add a **Lesson Plan Agent** that drafts lesson plans from the syllabus
- Replace `InMemorySessionService` with a database-backed service so quizzes persist between sessions
- Connect `email_agent` to the Gmail MCP server to actually send emails
- Add `ParallelAgent` to run syllabus analysis and timetable lookup simultaneously

---
*Google ADK docs: https://google.github.io/adk-docs/*